# 判別分析：受験生の合格・不合格を分類する

このNotebookでは、判別分析を使って、受験生の合格・不合格を分類する。

判別分析は、多変量解析の一種であり、複数の説明変数から、データがどのグループに属するかを判定する手法である。

## 1. ユースケース

教育機関や塾では、模試の点数だけでなく、面接評価、学習時間、出席率、課題評価などを総合的に見て、受験生の適性や合格可能性を判断する。

本Notebookでは、過去の受験生データをもとに、受験生が合格するか不合格になるかを分類する。

In [1]:
import pandas as pd

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

## 2. データ読み込み

受験生の模試成績、面接評価、学習時間、出席率、課題評価、合否結果を含むサンプルデータを読み込む。

In [2]:
df = pd.read_csv("../data/entrance_exam_sample.csv")
df

,mock_exam_score,interview_score,study_hours_per_week,attendance_rate,assignment_score,result
0,82,5,22,0.98,88,pass
1,76,4,18,0.95,82,pass
2,69,4,16,0.92,78,pass
3,91,5,25,0.99,94,pass
4,73,4,17,0.94,80,pass
5,64,3,12,0.88,70,fail
6,58,3,10,0.84,65,fail
7,49,2,7,0.78,55,fail
8,55,2,8,0.80,60,fail
9,62,3,11,0.86,68,fail


## 3. データ確認

データ型、基本統計量、合否結果の件数を確認する。

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   mock_exam_score       15 non-null     int64  
 1   interview_score       15 non-null     int64  
 2   study_hours_per_week  15 non-null     int64  
 3   attendance_rate       15 non-null     float64
 4   assignment_score      15 non-null     int64  
 5   result                15 non-null     str    
dtypes: float64(1), int64(4), str(1)
memory usage: 852.0 bytes


In [4]:
df.describe()

,mock_exam_score,interview_score,study_hours_per_week,attendance_rate,assignment_score
count,15.000000,15.000000,15.000000,15.000000,15.000000
mean,68.466667,3.533333,14.400000,0.892000,74.066667
std,12.540601,1.060099,5.913665,0.077201,12.360459
min,49.000000,2.000000,6.000000,0.760000,55.000000
25%,59.000000,3.000000,9.500000,0.830000,64.000000
50%,69.000000,4.000000,15.000000,0.920000,76.000000
75%,77.500000,4.000000,18.500000,0.955000,83.000000
max,91.000000,5.000000,25.000000,0.990000,94.000000


In [5]:
df["result"].value_counts()

result
pass    8
fail    7
Name: count, dtype: int64

## 4. 説明変数と目的変数

判別分析では、説明変数を使って目的変数のカテゴリを分類する。

今回の目的変数は `result` である。

In [7]:
X = df[
    [
        "mock_exam_score",
        "interview_score",
        "study_hours_per_week",
        "attendance_rate",
        "assignment_score",
    ]
]

y = df["result"]

## 5. 学習データとテストデータに分割

モデルの学習に使うデータと、評価に使うデータに分割する。

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y,
)

## 6. 線形判別分析の実行

線形判別分析を使って、合格者と不合格者を分ける境界を学習する。

In [9]:
model = LinearDiscriminantAnalysis()
model.fit(X_train, y_train)

,"solver solver: {'svd', 'lsqr', 'eigen'}, default='svd'Solver to use, possible values: - 'svd': Singular value decomposition (default). Does not compute the covariance matrix, therefore this solver is recommended for data with a large number of features. - 'lsqr': Least squares solution. Can be combined with shrinkage or custom covariance estimator. - 'eigen': Eigenvalue decomposition. Can be combined with shrinkage or custom covariance estimator... versionchanged:: 1.2 `solver=""svd""` now has experimental Array API support. See the :ref:`Array API User Guide ` for more details.",'svd'
,"shrinkage shrinkage: 'auto' or float, default=NoneShrinkage parameter, possible values: - None: no shrinkage (default). - 'auto': automatic shrinkage using the Ledoit-Wolf lemma. - float between 0 and 1: fixed shrinkage parameter.This should be left to None if `covariance_estimator` is used.Note that shrinkage works only with 'lsqr' and 'eigen' solvers.For a usage example, see:ref:`sphx_glr_auto_examples_classification_plot_lda.py`.",None
,"priors priors: array-like of shape (n_classes,), default=NoneThe class prior probabilities. By default, the class proportions areinferred from the training data.",None
,"n_components n_components: int, default=NoneNumber of components (<= min(n_classes - 1, n_features)) fordimensionality reduction. If None, will be set tomin(n_classes - 1, n_features). This parameter only affects the`transform` method.For a usage example, see:ref:`sphx_glr_auto_examples_decomposition_plot_pca_vs_lda.py`.",None
,"store_covariance store_covariance: bool, default=FalseIf True, explicitly compute the weighted within-class covariancematrix when solver is 'svd'. The matrix is always computedand stored for the other solvers... versionadded:: 0.17",False
,"tol tol: float, default=1.0e-4Absolute threshold for a singular value of X to be consideredsignificant, used to estimate the rank of X. Dimensions whosesingular values are non-significant are discarded. Only used ifsolver is 'svd'... versionadded:: 0.17",0.0001
,"covariance_estimator covariance_estimator: covariance estimator, default=NoneIf not None, `covariance_estimator` is used to estimatethe covariance matrices instead of relying on the empiricalcovariance estimator (with potential shrinkage).The object should have a fit method and a ``covariance_`` attributelike the estimators in :mod:`sklearn.covariance`.if None the shrinkage parameter drives the estimate.This should be left to None if `shrinkage` is used.Note that `covariance_estimator` works only with 'lsqr' and 'eigen'solvers... versionadded:: 0.24",None


## 7. テストデータで予測

学習済みモデルを使って、テストデータの合否を予測する。

In [10]:
y_pred = model.predict(X_test)

result_df = X_test.copy()
result_df["actual"] = y_test
result_df["predicted"] = y_pred

result_df

,mock_exam_score,interview_score,study_hours_per_week,attendance_rate,assignment_score,actual,predicted
4,73,4,17,0.94,80,pass,pass
1,76,4,18,0.95,82,pass,pass
8,55,2,8,0.80,60,fail,fail
14,79,4,19,0.96,84,pass,pass
13,53,2,6,0.76,58,fail,fail


In [11]:
accuracy_score(y_test, y_pred)

1.0

In [12]:
confusion_matrix(y_test, y_pred, labels=["pass", "fail"])

array([[3, 0],
       [0, 2]])

In [13]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

        fail       1.00      1.00      1.00         2
        pass       1.00      1.00      1.00         3

    accuracy                           1.00         5
   macro avg       1.00      1.00      1.00         5
weighted avg       1.00      1.00      1.00         5



## 9. 新しい受験生の判定

新しい受験生のデータを作成し、合格・不合格を予測する。

In [14]:
new_student = pd.DataFrame(
    [
        {
            "mock_exam_score": 75,
            "interview_score": 4,
            "study_hours_per_week": 18,
            "attendance_rate": 0.94,
            "assignment_score": 82,
        }
    ]
)

model.predict(new_student)

array(['pass'], dtype='<U4')

## 10. まとめ

このNotebookでは、模試の点数、面接評価、学習時間、出席率、課題評価を使って、受験生の合格・不合格を分類した。

判別分析では、目的変数が連続値ではなくカテゴリになる。

- 重回帰分析：点数や売上などの連続値を予測する
- 判別分析：合格・不合格などのカテゴリを分類する

今回の例では、複数の指標を使って合否を判別する流れを確認した。

なお、このデータは学習用のサンプルデータである。実際の教育現場で合否や適性を判断する場合は、モデルの結果だけで判断せず、公平性・説明可能性・個人情報保護に十分注意する必要がある。